**Correlation between Climate Variables and Density Variables**

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import openpyxl

df = pd.read_excel("file path")
df.columns = df.columns.str.strip()
print(df.columns.tolist())
df["Month"] = df["Month"].astype(str).str.strip()


In [ ]:

# ── COLUMN NAMES
density_cols = ["Non Vegetation", "Very Low", "Low", "Moderate", "High", "Very High"]

climate_cols = ["Mean_Relative_Humidity_pct", "Mean_Temperature_C", "SPI1_z_approx", "SPI3_z_approx", "Total_Rainfall_mm"]

density_labels = {
    "Non Vegetation": "Non Vegetation",
    "Very Low":       "Very Low",
    "Low":            "Low",
    "Moderate":       "Moderate",
    "High":           "High",
    "Very High":      "Very High"
}

climate_labels = {
    "Mean_Relative_Humidity_pct": "Mean Relative Humidity (%)",
    "Mean_Temperature_C":         "Mean Temperature (°C)",
    "SPI1_z_approx":              "SPI1 (Drought) 1-month",
    "SPI3_z_approx":              "SPI3 (Drought) 3-month",
    "Total_Rainfall_mm":          "Total Rainfall (mm)"
}

# ── Feb-May → DRY, Jun-Jan → WET 
df["Month"] = df["Month"].replace({"February": "DRY", "December": "WET"})
df["Month"] = df["Month"].replace({"March": "DRY", "September": "WET"})

# ── SELECT COLUMNS 
df = df[["Year", "Month"] + density_cols + climate_cols]

years  = sorted(df["Year"].unique())
months = ["DRY", "WET"]

# PRE-COMPUTE r & p
corr_lookup = {}
for m in months:
    group = df[df["Month"] == m]
    for d in density_cols:
        for c in climate_cols:
            if len(group) > 2:
                r, p = pearsonr(group[d], group[c])
                corr_lookup[(m, d, c)] = (round(r, 7),round(p, 7),round(r**2, 7)
)
            else:
                corr_lookup[(m, d, c)] = (np.nan, np.nan, np.nan)

# BUILD OUTPUT 
rows = []
for year in years:
    for month in months:
        for d in density_cols:
            for c in climate_cols:
                r, p, r_squared = corr_lookup[(month, d, c)]

                label = f"{density_labels[d]} vs {climate_labels[c]}"

                rows.append({
                "Year": int(year),
                "Comparison between": label,
                "Month": month,
                "correlation": r,
                "p values": p,
                "R square": r_squared
                })

output_df = pd.DataFrame(rows)
print(output_df.head().to_string(index=False))
print(f"\nTotal rows: {len(output_df)}")


# EXPORT 

output_df.to_excel(" file path",
                   index=False, engine="openpyxl")
print("Saved!")